In [1]:
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
import pandas as pd

In [2]:
def get_url(pageNum, auction=True, keywords="prizm+silver+psa10+basketball"):
    if auction:
        insert_mode = r"LH_Auction=1"
    else:
        insert_mode = r"LH_BIN=1"
    url_partA = r"https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&"+insert_mode+"&_nkw="+keywords
    url_partB = r"&_pgn="+str(pageNum)+"&_skc="+str(50*(pageNum-1))+"&rt=nc"
    url_partC = r"&_sacat=0"
    if pageNum == 1:
        url = url_partA + url_partC
    else:
        url = url_partA + url_partB
    return url

----

**Download Auctions**

In [3]:
data = defaultdict(list)
for pageNum in range(1, 20):
    url = get_url(pageNum)
    print(url)
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    items = soup.findAll(class_='sresult lvresult clearfix li')
    print(r"Page "+str(pageNum)+r": "+str(len(items)))
    for item in items:
        time = item.find(class_='tme').contents[1].contents[0]
        title = item.find(class_='lvtitle').find(class_='vip').contents[0]
        item_url = item.find(class_='lvtitle').find(class_='vip')['href']
        price = item.find(class_='lvprice prc').find(class_="bold bidsold").contents[0]
        price = float(str(price).replace("$", "").replace(",", ""))
        bids = item.find(class_='lvformat').contents[1].contents[0]
        bids = int(bids.split(" ")[0])

        data["Timestamp"].append(time)
        data["Title"].append(title)
        data["URL"].append(item_url)
        data["Price"].append(price)
        data["Bids"].append(bids)

https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_Auction=1&_nkw=prizm+silver+psa10+basketball&_sacat=0
Page 1: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_Auction=1&_nkw=prizm+silver+psa10+basketball&_pgn=2&_skc=50&rt=nc
Page 2: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_Auction=1&_nkw=prizm+silver+psa10+basketball&_pgn=3&_skc=100&rt=nc
Page 3: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_Auction=1&_nkw=prizm+silver+psa10+basketball&_pgn=4&_skc=150&rt=nc
Page 4: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_u

In [4]:
auction_data = pd.DataFrame(data).set_index("Timestamp")
auction_data.head()

,Bids,Price,Title,URL
Timestamp,,,,
Mar-18 17:37,7,36.20,2017-18 Panini Prizm Silver Ben Simmons #9 PSA...,https://www.ebay.com/itm/2017-18-Panini-Prizm-...
Mar-18 17:36,9,37.00,2017-18 Panini Prizm Silver Ben Simmons #9 PSA...,https://www.ebay.com/itm/2017-18-Panini-Prizm-...
Mar-18 17:31,5,26.01,2017-18 Panini Prizm Silver Ben Simmons #9 PSA...,https://www.ebay.com/itm/2017-18-Panini-Prizm-...
Mar-18 17:11,14,34.00,2017-18 Panini Prizm Silver Ben Simmons #9 PSA...,https://www.ebay.com/itm/2017-18-Panini-Prizm-...
Mar-18 17:06,8,39.00,2017-18 Panini Prizm Silver Ben Simmons #9 PSA...,https://www.ebay.com/itm/2017-18-Panini-Prizm-...


In [ ]:
auction_data.to_csv(r"../output/auction_data.csv")

----

**Download "Buy-it-now"**

In [ ]:
raw_data = defaultdict(list)
for pageNum in range(1, 20):
    url = get_url(pageNum, auction=False)
    print(url)
    page = requests.get(url)
    soup = BeautifulSoup(page.text, 'html.parser')
    items = soup.findAll(class_='sresult lvresult clearfix li')
    print(r"Page "+str(pageNum)+r": "+str(len(items)))
    for item in items:
        title = item.find(class_='lvtitle').find(class_='vip').contents[0]
        try:
            price = item.find(class_='lvprice prc').find(class_="bold bidsold").find(class_="sboffer").contents[0]
        except AttributeError:
            price = item.find(class_='lvprice prc').find(class_="bold bidsold").contents[0]
        price = float(str(price).replace("$", "").replace(",", ""))
        item_url = item.find(class_='lvtitle').find(class_='vip')['href']
        item_page = requests.get(item_url)
        item_soup = BeautifulSoup(item_page.text, 'html.parser')
        
        try:
            date = item_soup.find(id="bb_tlft").contents[0].replace(r"/n", "").replace(r"/t", "")
        except AttributeError:
            date = "N/A"

        raw_data["Timestamp"].append(date)
        raw_data["Title"].append(title)
        raw_data["URL"].append(item_url)
        raw_data["Price"].append(price)
        raw_data["Bids"].append("N/A")

https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_BIN=1&_nkw=prizm+silver+psa10+basketball&_sacat=0
Page 1: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_BIN=1&_nkw=prizm+silver+psa10+basketball&_pgn=2&_skc=50&rt=nc
Page 2: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_BIN=1&_nkw=prizm+silver+psa10+basketball&_pgn=3&_skc=100&rt=nc
Page 3: 50
https://www.ebay.com/sch/i.html?_from=R40&_sacat=0&LH_Sold=1&_udlo=&_udhi=&_samilow=&_samihi=&_sadis=15&_stpos=07014&_sop=13&_dmd=1&LH_Complete=1&_fosrp=1&LH_BIN=1&_nkw=prizm+silver+psa10+basketball&_pgn=4&_skc=150&rt=nc
Page 4: 50


ProxyError: HTTPSConnectionPool(host='www.ebay.com', port=443): Max retries exceeded with url: /itm/2019-20-Panini-Prizm-Silver-288-Darius-Garland-Cavaliers-RC-Rookie-PSA-10/333543768323?hash=item4da8c20103:g:6xwAAOSw3g5eaoj3 (Caused by ProxyError('Cannot connect to proxy.', RemoteDisconnected('Remote end closed connection without response',)))

In [ ]:
buy_it_now = pd.DataFrame(raw_data).set_index("Timestamp")
buy_it_now.head()

In [ ]:
buy_it_now.to_csv(r"../output/buy_it_now.csv")

In [ ]:
buy_it_now.head()